# TF-IDF & Cosine Similarity — Class Activity

**Student:** Taha Abbas
**Roll No:** 26I-8004
**Course:** Data Science Tools and Techniques 

This notebook completes every step in `task_to_do.txt`:

1. Load the 10 long-text documents and check their word counts
2. Build the TF-IDF matrix with `stop_words="english"`
3. Inspect the matrix as a DataFrame and find the top-5 terms per document
4. Rank documents against three queries using cosine similarity
5. Optional challenge: bigrams with `max_df` / `min_df` filtering
6. Verify the work with the provided unit tests

## Setup

Finds the repository root so that `src/` and `data/` can be imported. This works whether the notebook is run locally from the repo folder or in Google Colab after cloning the repo to `/content`.

In [6]:
import os
import sys

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def find_repo_root():
    candidates = [os.getcwd(), os.path.dirname(os.getcwd()), "/content/unstructuredata_tfidf_repo"]
    for path in candidates:
        if os.path.isfile(os.path.join(path, "src", "load_data.py")):
            return path
    raise FileNotFoundError("Could not find the repository. Run this notebook from inside the repo folder.")


REPO_ROOT = find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.load_data import load_documents

pd.set_option("display.width", 120)
print("Repository root:", REPO_ROOT)

Repository root: /home/veroke/Documents/Projects/FAST-NUCES/DSTT/unstructuredata_tfidf_repo


## Step 1 — Load the Data

Load the 10 titles and their documents, confirm that exactly 10 were loaded, and check the word count of each one.

In [7]:
titles, docs = load_documents()

print(f"{len(docs)} documents loaded.")
assert len(titles) == len(docs) == 10, "Expected exactly 10 documents"

word_counts = pd.DataFrame(
    {"Words": [len(d.split()) for d in docs]},
    index=pd.Index(titles, name="Document"),
)
print(word_counts)
print(f"\nShortest: {word_counts['Words'].min()} words, longest: {word_counts['Words'].max()} words")
print("Every document is long text (> 30 words):", bool((word_counts["Words"] > 30).all()))

10 documents loaded.
                      Words
Document                   
AI in Healthcare         85
Climate Change           83
Basketball Strategy      85
Italian Pasta            91
Space Exploration        86
Cybersecurity            88
Music Theory             87
Gardening Vegetables     89
Economics Inflation      89
Public Health            91

Shortest: 83 words, longest: 91 words
Every document is long text (> 30 words): True


## Step 2 — Build the TF-IDF Matrix

`TfidfVectorizer` lowercases and tokenizes each document, removes English stop words, and gives each term a weight of **TF × IDF**:

- **TF** (term frequency): how many times the term appears in the document.
- **IDF** (inverse document frequency): `ln((1 + N) / (1 + df)) + 1`, where `N` is the number of documents and `df` is how many documents contain the term. Rare terms get a higher IDF.

Each row is then L2-normalized to length 1, so long and short documents are comparable.

In [8]:
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(docs)
feature_names = vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(feature_names))

n_cells = tfidf_matrix.shape[0] * tfidf_matrix.shape[1]
print(f"Non-zero entries: {tfidf_matrix.nnz} of {n_cells} ({tfidf_matrix.nnz / n_cells:.1%}), so the matrix is sparse")

TF-IDF matrix shape: (10, 537)
Vocabulary size: 537
Non-zero entries: 620 of 5370 (11.5%), so the matrix is sparse


## Step 3 — Inspect the Matrix as a DataFrame

Convert the sparse matrix into a DataFrame with document titles as rows and terms as columns.

In [9]:
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), index=titles, columns=feature_names)

print("Sample of the TF-IDF matrix (first 3 documents x first 10 terms):")
df_tfidf.iloc[:3, :10].round(3)

Sample of the TF-IDF matrix (first 3 documents x first 10 terms):


,ability,accelerate,access,accountability,accuracy,accurate,actions,activity,adaptation,add
AI in Healthcare,0.00,0.0,0.0,0.125,0.125,0.000,0.000,0.0,0.000,0.0
Climate Change,0.00,0.0,0.0,0.000,0.000,0.125,0.125,0.0,0.125,0.0
Basketball Strategy,0.13,0.0,0.0,0.000,0.000,0.000,0.000,0.0,0.000,0.0


### Top-5 TF-IDF terms per document

The highest-scoring terms are those that appear often in one document but rarely in the others, so they act as a short summary of that document.

In [10]:
def top_terms(df, n=5):
    return {title: df.loc[title].nlargest(n) for title in df.index}


top5 = top_terms(df_tfidf, n=5)
for title, terms in top5.items():
    print(f"{title}:")
    print("   " + ", ".join(f"{term} ({score:.3f})" for term, score in terms.items()))

AI in Healthcare:
   ai (0.374), healthcare (0.249), use (0.165), accountability (0.125), accuracy (0.125)
Climate Change:
   change (0.250), emissions (0.250), climate (0.212), energy (0.212), accurate (0.125)
Basketball Strategy:
   ability (0.130), analytics (0.130), basketball (0.130), breaks (0.130), chemistry (0.130)
Italian Pasta:
   pasta (0.443), sauces (0.332), sauce (0.222), different (0.188), add (0.111)
Space Exploration:
   space (0.349), mars (0.233), missions (0.233), accelerate (0.116), advanced (0.116)
Cybersecurity:
   cybersecurity (0.249), systems (0.212), data (0.165), attackers (0.125), attacks (0.125)
Music Theory:
   music (0.247), musicians (0.247), practice (0.210), allows (0.123), artists (0.123)
Gardening Vegetables:
   soil (0.358), pests (0.239), year (0.239), improves (0.203), adds (0.119)
Economics Inflation:
   inflation (0.235), spending (0.235), economic (0.199), trade (0.199), affect (0.117)
Public Health:
   health (0.346), public (0.233), sleep (0

**Observation:** For most documents the top terms clearly identify the topic, for example `pasta` / `sauces` for Italian Pasta, `space` / `mars` / `missions` for Space Exploration, and `soil` / `pests` for Gardening Vegetables.

Because each document is short (about 85–90 words), most words appear only once. Many terms therefore end up with the same score. Basketball Strategy shows this best: its top 5 are tied, and which tied terms make the list depends on alphabetical order rather than importance.

## Step 4 — Query Similarity Challenge

The query is transformed with the **same fitted vectorizer**, so it uses the same vocabulary and IDF weights as the documents. Then cosine similarity is computed against all 10 documents:

$$\cos(q, d) = \frac{q \cdot d}{\lVert q \rVert \, \lVert d \rVert}$$

A score of 1 means identical direction, and 0 means no shared terms.

In [11]:
def rank_documents(query, vectorizer, tfidf_matrix, titles):
    """Return [(title, score)] for all documents, sorted by cosine similarity (highest first)."""
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    return sorted(zip(titles, scores), key=lambda x: x[1], reverse=True)


expected = {
    "renewable energy and climate policy": "Climate Change",
    "password security and phishing": "Cybersecurity",
    "healthy sleep exercise and diet": "Public Health",
}

for query, expected_title in expected.items():
    ranking = rank_documents(query, vectorizer, tfidf_matrix, titles)
    print(f"Query: {query!r}")
    for rank, (title, score) in enumerate(ranking, start=1):
        print(f"  {rank:>2}. {title:<22} {score:.4f}")
    top_title = ranking[0][0]
    status = "PASS" if top_title == expected_title else "FAIL"
    print(f"  -> Expected '{expected_title}' first, got '{top_title}': {status}\n")

Query: 'renewable energy and climate policy'
   1. Climate Change         0.3235
   2. Economics Inflation    0.0952
   3. Gardening Vegetables   0.0485
   4. AI in Healthcare       0.0000
   5. Basketball Strategy    0.0000
   6. Italian Pasta          0.0000
   7. Space Exploration      0.0000
   8. Cybersecurity          0.0000
   9. Music Theory           0.0000
  10. Public Health          0.0000
  -> Expected 'Climate Change' first, got 'Climate Change': PASS

Query: 'password security and phishing'
   1. Cybersecurity          0.1764
   2. AI in Healthcare       0.0000
   3. Climate Change         0.0000
   4. Basketball Strategy    0.0000
   5. Italian Pasta          0.0000
   6. Space Exploration      0.0000
   7. Music Theory           0.0000
   8. Gardening Vegetables   0.0000
   9. Economics Inflation    0.0000
  10. Public Health          0.0000
  -> Expected 'Cybersecurity' first, got 'Cybersecurity': PASS

Query: 'healthy sleep exercise and diet'
   1. Public Health     

### Why do the scores look like this?

A query term can only contribute if it is in the vocabulary that was learned from the documents. The check below shows which query words the model actually knows.

In [12]:
vocab = vectorizer.vocabulary_
stop_words = vectorizer.get_stop_words()

for query in expected:
    print(f"Query: {query!r}")
    for word in query.split():
        if word in stop_words:
            status = "removed (stop word)"
        elif word in vocab:
            status = f"in vocabulary (IDF = {vectorizer.idf_[vocab[word]]:.2f})"
        else:
            status = "NOT in vocabulary -> ignored"
        print(f"   {word:<10} {status}")
    print()

Query: 'renewable energy and climate policy'
   renewable  in vocabulary (IDF = 2.70)
   energy     in vocabulary (IDF = 2.30)
   and        removed (stop word)
   climate    in vocabulary (IDF = 2.30)
   policy     in vocabulary (IDF = 2.30)

Query: 'password security and phishing'
   password   NOT in vocabulary -> ignored
   security   in vocabulary (IDF = 2.70)
   and        removed (stop word)
   phishing   in vocabulary (IDF = 2.70)

Query: 'healthy sleep exercise and diet'
   healthy    in vocabulary (IDF = 2.30)
   sleep      in vocabulary (IDF = 2.70)
   exercise   in vocabulary (IDF = 2.70)
   and        removed (stop word)
   diet       in vocabulary (IDF = 2.70)



**Observations:**

- All three queries rank the expected document first.
- `password` is ignored because it is not in the vocabulary. The Cybersecurity document uses the plural "passwords", and TF-IDF has no stemming, so the two forms are different tokens. That is why the Cybersecurity score is lower than the other two top scores.
- Most other documents score exactly 0 because they share no terms with the query. TF-IDF only matches exact words, not meaning. The small non-zero scores come from one or two shared words: Economics Inflation shares `energy` and `policy` with the climate query, and Gardening Vegetables shares `climate` with the climate query and `healthy` with the health query.

## Step 5 — Optional Challenge: Bigrams with `max_df` / `min_df`

- `ngram_range=(1, 2)` adds two-word phrases (bigrams) to the vocabulary.
- `max_df=0.8` drops terms that appear in more than 80% of the documents.
- `min_df=2` drops terms that appear in fewer than 2 documents.

In [13]:
vectorizer_v2 = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.8,
    min_df=2,
)
tfidf_matrix_v2 = vectorizer_v2.fit_transform(docs)
feature_names_v2 = vectorizer_v2.get_feature_names_out()
df_tfidf_v2 = pd.DataFrame(tfidf_matrix_v2.toarray(), index=titles, columns=feature_names_v2)

bigrams = [term for term in feature_names_v2 if " " in term]

print("Vocabulary size")
print(f"  Original (unigrams):        {len(feature_names)}")
print(f"  Challenge (1-2 grams, df):  {len(feature_names_v2)}")
print(f"\nBigrams that survived min_df=2: {len(bigrams)} -> {bigrams}")

Vocabulary size
  Original (unigrams):        537
  Challenge (1-2 grams, df):  69

Bigrams that survived min_df=2: 3 -> ['economic growth', 'policy involves', 'trade offs']


### Comparing the top terms

In [14]:
top5_v2 = top_terms(df_tfidf_v2, n=5)

comparison = pd.DataFrame(
    {
        "Original top-5": [", ".join(top5[t].index) for t in titles],
        "Challenge top-5": [", ".join(top5_v2[t].index) for t in titles],
    },
    index=pd.Index(titles, name="Document"),
)
with pd.option_context("display.max_colwidth", None):
    display(comparison)

,Original top-5,Challenge top-5
Document,,
AI in Healthcare,"ai, healthcare, use, accountability, accuracy","use, careful, clean, disease, improve"
Climate Change,"change, emissions, climate, energy, accurate","climate, energy, economic, economic growth, efficiency"
Basketball Strategy,"ability, analytics, basketball, breaks, chemistry","adjust, based, communication, decisions, defenses"
Italian Pasta,"pasta, sauces, sauce, different, add","different, flavor, improve, long, quality"
Space Exploration,"space, mars, missions, accelerate, advanced","international, medical, health, data, water"
Cybersecurity,"cybersecurity, systems, data, attackers, attacks","systems, data, defenses, essential, intelligence"
Music Theory,"music, musicians, practice, allows, artists","practice, communication, different, improves, live"
Gardening Vegetables,"soil, pests, year, improves, adds","improves, careful, climate, flavor, healthy"
Economics Inflation,"inflation, spending, economic, trade, affect","economic, trade, adjust, decisions, economic growth"


### Do the queries still work with the challenge vectorizer?

In [15]:
for query, expected_title in expected.items():
    ranking = rank_documents(query, vectorizer_v2, tfidf_matrix_v2, titles)
    top_title, top_score = ranking[0]
    known = [w for w in query.split() if w in vectorizer_v2.vocabulary_]
    verdict = "correct" if top_title == expected_title else f"wrong, expected {expected_title}"
    print(f"Query: {query!r}")
    print(f"   terms still in vocabulary: {known}")
    print(f"   top document: {top_title} ({top_score:.4f}) -> {verdict}\n")

Query: 'renewable energy and climate policy'
   terms still in vocabulary: ['energy', 'climate', 'policy']
   top document: Climate Change (0.5575) -> correct

Query: 'password security and phishing'
   terms still in vocabulary: []
   top document: AI in Healthcare (0.0000) -> wrong, expected Cybersecurity

Query: 'healthy sleep exercise and diet'
   terms still in vocabulary: ['healthy']
   top document: Gardening Vegetables (0.2180) -> wrong, expected Public Health



### Do bigrams make the terms more meaningful?

**Not on this dataset. The terms become less meaningful.**

- The vocabulary shrinks sharply because `min_df=2` removes every term that appears in only one document. With 10 documents on 10 unrelated topics, the words found in a single document are exactly the topic words (`pasta`, `mars`, `phishing`, `basketball`), so they are the ones that get removed.
- Only a few bigrams survive, because almost no two-word phrase is repeated across different topics. The survivors (for example `economic growth`) are not specific to one document.
- The remaining top terms are generic words shared between topics (`different`, `practice`, `data`, `systems`). For example, Italian Pasta's top term is now `different`, and Space Exploration's top terms include `medical` and `health`.
- Retrieval gets worse. Only the climate query still ranks the right document first. `phishing`, `sleep`, `exercise` and `diet` were removed, so the cybersecurity query matches nothing (all scores 0) and the health query ranks Gardening Vegetables first through the shared word `healthy`.

Bigrams together with `min_df` are useful on a **large** corpus where many documents share a topic. In that case, phrases like "climate change" or "machine learning" repeat across documents and carry more meaning than the single words. For a small corpus of 10 very different documents, the default unigram settings work better.

## Step 6 — Verify the Work

Run the provided unit tests (`pytest -q`). They check that:

- 10 documents are loaded and each is long text (> 30 words)
- the TF-IDF matrix has 10 rows and as many columns as the vocabulary size
- the query "renewable energy and climate policy" ranks Climate Change first

In [16]:
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no", "tests"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout or result.stderr)
assert result.returncode == 0, "Some tests failed"

...                                                                      [100%]
3 passed in 1.54s



## Conclusion

| Step | Result |
|---|---|
| 1. Load data | 10 documents loaded, each about 85–90 words |
| 2. TF-IDF matrix | Shape (10, 537), vocabulary of 537 terms |
| 3. Top terms | Top-5 terms summarize each document's topic |
| 4. Query ranking | All 3 queries rank the expected document first |
| 5. Bigram challenge | Vocabulary drops to 69 terms (only 3 bigrams), topic words are lost and 2 of 3 queries fail, so bigrams do not help on this small corpus |
| 6. Unit tests | All 3 tests pass |

### Exit ticket

1. **What does a high TF-IDF score mean for a term in one document?**
   The term appears often in that document but rarely in the others, so it is characteristic of that document.

2. **Why does IDF reduce the weight of common words like "the" or "data"?**
   A word that appears in many documents does not help tell them apart. IDF gives such words a lower weight. For example, `data` appears in several documents here and has a lower IDF than a word like `soil`, which appears in only one.

3. **Name one real task where TF-IDF is useful.**
   Search engines and document retrieval: ranking documents or web pages against a search query, as done in Step 4. Other uses include keyword extraction and features for text classification such as spam filtering.

### Limitations of TF-IDF

- **No semantics:** synonyms and related words ("healthy" and "health", "password" and "passwords") are treated as unrelated tokens.
- **No word order:** it is a bag of words, so "dog bites man" and "man bites dog" get the same vector.
- **Exact vocabulary only:** words in a query that never appeared in the documents are ignored.